## Train&Test Model

In [ ]:
import pandas as pd
import random
from fastai.vision.all import *

# --- 1. ตั้งค่า PATH ---
EXTERNAL_DATA_PATH = Path('/kaggle/input/ai-generated-images-vs-real-images')
MY_DATA_PATH = Path('/kaggle/input/ai-vs-real-testset/test')

# --- 2. ฟังก์ชัน Labeling ---
def get_label(o):
    path_str = str(o).lower()
    # เช็คทั้งจากโฟลเดอร์ Dataset นอก และชื่อไฟล์ของเรา
    if 'aiartdata' in path_str or 'gemini' in path_str or 'seedream' in path_str:
        return 'AI_FAKE'
    return 'HUMAN_REAL'

print("⏳ [1/5] เริ่มดึงไฟล์แบบละเอียด...")

# --- 3. ดึงไฟล์ (เพิ่ม recursive=True เพื่อความชัวร์) ---
ext_files = get_image_files(EXTERNAL_DATA_PATH, recurse=True)
my_files = get_image_files(MY_DATA_PATH, recurse=True)

# กรองไฟล์ Dataset นอก
ext_ai = [f for f in ext_files if 'AiArtData' in str(f)]
ext_real = [f for f in ext_files if 'RealArt' in str(f)]

# กรองไฟล์ของเรา (ใช้ str(f).lower() เพื่อเช็คทั้ง Path และชื่อไฟล์)
my_fake = [f for f in my_files if 'gemini' in str(f).lower() or 'seedream' in str(f).lower()]
my_real = [f for f in my_files if f not in my_fake]

# --- จุด Checkpoint: ถ้ายังเป็น 0 ระบบจะแจ้งเตือนทันที ---
print(f"🔍 สแกนเจอไฟล์:")
print(f"- AI จากเน็ต: {len(ext_ai)} รูป")
print(f"- REAL จากเน็ต: {len(ext_real)} รูป")
print(f"- ของคุณ (AI): {len(my_fake)} รูป")
print(f"- ของคุณ (REAL): {len(my_real)} รูป")

if len(my_fake) == 0:
    print("❌ ERROR: ยังหาไฟล์ AI ของคุณไม่เจอ! ลองเช็คชื่อไฟล์ 3 รูปแรก:")
    for f in my_files[:3]: print(f"-> {f}")
    raise ValueError("Stop: AI files not found.")

# --- 4. จัดสรรข้อมูล (เทรน 50% / เทส 50%) ---
# แบ่งรูปเราไป Test อย่างละ 500
test_size_ai = min(500, len(my_fake))
test_size_real = min(500, len(my_real))

test_files = L(random.sample(my_fake, test_size_ai) + random.sample(my_real, test_size_real))

# ส่วนที่เหลือเอาไปสอน (Train)
train_my_fake = [f for f in my_fake if f not in test_files]
train_my_real = [f for f in my_real if f not in test_files]

# รวมไฟล์เทรน (ของนอก + ของเราส่วนที่เหลือ)
final_train_files = L(ext_ai + ext_real + train_my_fake + train_my_real)

print(f"\n📊 สรุปยอดสุดท้าย:")
print(f"- ใช้เทรน: {len(final_train_files)} รูป")
print(f"- เก็บไว้เทส (Unseen): {len(test_files)} รูป")

# --- 5. สร้าง DataLoaders และเทรน ---
dls = ImageDataLoaders.from_path_func(
    EXTERNAL_DATA_PATH, 
    final_train_files, 
    get_label,
    valid_pct=0.2,
    item_tfms=Resize(224),
    batch_tfms=aug_transforms(mult=1.5),
    bs=16
)

print("\n⏳ [2/5] เริ่มเทรน ResNet50 (6 Epochs)...")
learn = vision_learner(dls, resnet50, metrics=[accuracy, error_rate])
learn.fine_tune(6)

# --- 6. สรุปผล ---
print("\n⏳ [3/5] กำลังทดสอบกับ Unseen Test...")
test_dl = learn.dls.test_dl(test_files)
probs, _ = learn.get_preds(dl=test_dl)
preds = probs.argmax(dim=1)

results = []
for i in range(len(test_files)):
    fname = test_files[i].name.lower()
    pred_label = dls.vocab[preds[i]]
    source = 'Gemini (AI)' if 'gemini' in fname else 'Seedream (AI)' if 'seedream' in fname else 'Real Human (Real)'
    results.append({'File': test_files[i].name, 'Source': source, 'Detected_As': pred_label})

df_final = pd.DataFrame(results)
print("\n🎯 [4/5] สรุปผลลัพธ์:")
print(pd.crosstab(df_final['Source'], df_final['Detected_As']))
df_final.to_csv('final_pro_report.csv', index=False)

## Summarize the result

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report

# 1. โหลดข้อมูล (เปลี่ยนชื่อไฟล์ให้ตรงกับที่คุณเซฟไว้ล่าสุด)
df = pd.read_csv('final_pro_report.csv')

# 2. แปลงแหล่งที่มา (Actual) ให้เป็น Class เดียวกับที่โมเดลทาย (Predicted)
# เราจะจัดกลุ่ม Gemini และ Seedream ให้เป็น 'AI_FAKE'
df['Actual_Label'] = df['Source'].apply(lambda x: 'AI_FAKE' if 'AI' in str(x) else 'HUMAN_REAL')

# 3. คำนวณ Confusion Matrix
labels = ['AI_FAKE', 'HUMAN_REAL']
cm = confusion_matrix(df['Actual_Label'], df['Detected_As'], labels=labels)

# 4. พล็อต Heatmap ให้สวยงาม
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='RdPu', # สีชมพู-ม่วง ดูทันสมัยสไตล์ Fintech
            xticklabels=['Predicted AI', 'Predicted Human'], 
            yticklabels=['Actual AI', 'Actual Human'])

plt.title('AI vs Human Detection: Confusion Matrix', fontsize=15, pad=20)
plt.ylabel('Reality (Actual Data)', fontsize=12)
plt.xlabel('Model Decision (Predicted)', fontsize=12)
plt.savefig('ai_confusion_matrix.png', dpi=300) # เซฟรูปไปใส่สไลด์ได้เลย
plt.show()

# 5. พิมพ์สรุปค่า Precision, Recall, F1-Score
print("\n📋 --- Detailed Classification Report ---")
print(classification_report(df['Actual_Label'], df['Detected_As'], target_names=labels))

## Reveal the pictures that being classified in the wrong group

In [ ]:
# ดูรูป AI ที่หลอกโมเดลได้เนียนที่สุด (Top 5 Confidence)
top_ai_fakes = df_errors[df_errors['actual'] == 'AI_FAKE'].sort_values(by='confidence', ascending=False)
print(top_ai_fakes[['filename', 'confidence', 'ai_source']].head(5))

# ดูรูปที่โมเดลลังเลที่สุด (Confidence ต่ำสุด)
most_uncertain = df_errors.sort_values(by='confidence', ascending=True)
print(most_uncertain[['filename', 'confidence', 'actual']].head(5))

In [ ]:
# 1. กรองเฉพาะกรณีที่ Actual เป็น AI แต่ Predicted เป็น Human
# (ใช้ df_results จากโค้ดก่อนหน้าที่เราสร้างไว้)
df_ai_fails = df_results[
    (df_results['actual'] == 'AI_FAKE') & 
    (df_results['predicted'] == 'HUMAN_REAL')
].copy()

# 2. เพิ่มคอลัมน์ระบุค่ายเพื่อความง่ายในการวิเคราะห์
def get_source(fname):
    fname = fname.lower()
    if 'seedream' in fname: return 'Seedream'
    if 'gemini' in fname: return 'Gemini'
    return 'Other AI'

df_ai_fails['ai_source'] = df_ai_fails['filename'].apply(get_source)

# 3. เรียงลำดับตาม Confidence จากมากไปน้อย (รูปที่เนียนที่สุดจะอยู่บนสุด)
df_ai_fails = df_ai_fails.sort_values(by='confidence', ascending=False)

# 4. เซฟเป็นไฟล์ CSV
df_ai_fails.to_csv('ai_misclassified_as_human.csv', index=False)

print(f"✅ สร้างไฟล์ ai_misclassified_as_human.csv สำเร็จ! (จำนวน {len(df_ai_fails)} รูป)")
print(df_ai_fails[['filename', 'confidence', 'ai_source']].head(10))

In [ ]:
# 1. กรองเฉพาะรูปที่เป็น Gemini แต่โมเดลทายว่าเป็น HUMAN_REAL
df_gemini_fakes = df_all_results[
    (df_all_results['ai_source'] == 'Gemini') & 
    (df_all_results['predicted'] == 'HUMAN_REAL')
].copy()

# 2. เรียงลำดับตาม Confidence จากมากไปน้อย (Descending)
# รูปที่อยู่บนสุดคือรูปที่โมเดล "ปักใจเชื่อ" ที่สุดว่าเป็นคนจริง
df_gemini_fakes = df_gemini_fakes.sort_values(by='confidence', ascending=False)

print("🎯 5 อันดับรูป Gemini ที่เนียนที่สุด (โมเดลมั่นใจว่าเป็นคนจริงสูงที่สุด):")
print(df_gemini_fakes[['filename', 'confidence']].head(5))

# 3. เก็บชื่อไฟล์อันดับ 1 ไว้ไปเปิดดูรูป
if not df_gemini_fakes.empty:
    top_gemini_fake = df_gemini_fakes.iloc[0]['filename']
    top_confidence = df_gemini_fakes.iloc[0]['confidence']
    print(f"\n🔥 รูปที่เนียนที่สุดคือ: {top_gemini_fake} ด้วยความมั่นใจ {top_confidence:.4f}")